In [14]:
import pandas as pd
import numpy as np
import joblib
import glob
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

# 1. TẢI VÀ GỘP DỮ LIỆU PARQUET
print("[*] Đang đọc các file .parquet từ thư mục data/archive...")
file_paths = glob.glob('../data/archive/*.parquet')

df_list = []
for file in file_paths:
    temp_df = pd.read_parquet(file)
    df_list.append(temp_df)

df = pd.concat(df_list, ignore_index=True)
print(f"[*] Tổng số dòng dữ liệu ban đầu: {len(df)}")
print(df.columns.tolist())
# 2. XỬ LÝ NHÃN (CTU-13 Specific Labeling)
# Trong CTU-13, Label thường có dạng "flow=Background", "flow=To-Normal", "flow=From-Botnet..."
def categorize_label(label_str):
    label_str = str(label_str).lower()
    if 'botnet' in label_str:
        return 1 # Malicious
    elif 'normal' in label_str:
        return 0 # Benign
    else:
        return -1 # Background (Sẽ bị loại bỏ để mô hình tập trung phân biệt rõ ràng)

df['Target'] = df['label'].apply(categorize_label)

# Chỉ giữ lại Botnet (1) và Normal (0), vứt bỏ Background (-1)
df = df[df['Target'] != -1]



[*] Đang đọc các file .parquet từ thư mục data/archive...
[*] Tổng số dòng dữ liệu ban đầu: 10598771
['dur', 'proto', 'dir', 'state', 'stos', 'dtos', 'tot_pkts', 'tot_bytes', 'src_bytes', 'label', 'Family']


In [16]:
# 3. LỰA CHỌN ĐẶC TRƯNG CHUẨN (Feature Selection)
# TUYỆT ĐỐI KHÔNG đưa IP (SrcAddr, DstAddr) hay Port (Sport, Dport) vào train
# để tránh Random Forest học vẹt mục tiêu thay vì học "hành vi"
features = [
    'dur',        # Flow Duration
    'tot_pkts',    # Total Packets
    'tot_bytes',   # Total Bytes
    'src_bytes',   # Source Bytes
    'proto',      # Protocol (cần encode)
    'state'       # Connection State (cần encode)
]

X = df[features].copy()
y = df['Target'].copy()

# Xóa các dòng có giá trị NaN
valid_indices = X.dropna().index
X = X.loc[valid_indices]
y = y.loc[valid_indices]

# Mã hóa các cột phân loại (Categorical) thành số
le_proto = LabelEncoder()
X['proto'] = le_proto.fit_transform(X['proto'].astype(str))

le_state = LabelEncoder()
X['state'] = le_state.fit_transform(X['state'].astype(str))

# 4. CÂN BẰNG DỮ LIỆU (Undersampling)
print(f"[*] Phân bố trước khi cân bằng: \n{y.value_counts()}")
botnet_indices = y[y == 1].index
normal_indices = y[y == 0].index

# Lấy ngẫu nhiên mẫu Normal bằng với số lượng Botnet
min_samples = min(len(botnet_indices), len(normal_indices))
random_normal_indices = np.random.choice(normal_indices, min_samples, replace=False)
random_botnet_indices = np.random.choice(botnet_indices, min_samples, replace=False)

balanced_indices = np.concatenate([random_botnet_indices, random_normal_indices])
X_balanced = X.loc[balanced_indices]
y_balanced = y.loc[balanced_indices]

print(f"[*] Phân bố sau khi cân bằng: \n{y_balanced.value_counts()}")

# 5. HUẤN LUYỆN RANDOM FOREST
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced, y_balanced, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_balanced
)

print("[*] Đang huấn luyện The Surrogate Judge (Random Forest)...")
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_train)

# 6. ĐÁNH GIÁ & LƯU MÔ HÌNH
y_pred = rf_clf.predict(X_test)
print("\n[+] KẾT QUẢ ĐÁNH GIÁ (Classification Report):")
print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'Botnet (1)']))

joblib.dump(rf_clf, '../data/surrogate_ids_ctu13.pkl')
joblib.dump(le_proto, '../data/label_encoder_proto.pkl')
joblib.dump(le_state, '../data/label_encoder_state.pkl')
print("[+] Đã lưu mô hình và các bộ mã hóa (Encoders) thành công!")

[*] Phân bố trước khi cân bằng: 
Target
1    262504
0    202549
Name: count, dtype: int64
[*] Phân bố sau khi cân bằng: 
Target
1    202549
0    202549
Name: count, dtype: int64
[*] Đang huấn luyện The Surrogate Judge (Random Forest)...

[+] KẾT QUẢ ĐÁNH GIÁ (Classification Report):
              precision    recall  f1-score   support

  Normal (0)       0.93      0.94      0.94     40510
  Botnet (1)       0.94      0.93      0.94     40510

    accuracy                           0.94     81020
   macro avg       0.94      0.94      0.94     81020
weighted avg       0.94      0.94      0.94     81020

[+] Đã lưu mô hình và các bộ mã hóa (Encoders) thành công!


In [17]:
# 3. TÁCH TẬP MALICIOUS (BOTNET)
malicious_df = df[df['Target'] == 1].copy()

print(f"[+] Số mẫu malicious: {len(malicious_df):,}")
print(f"[+] Số mẫu normal:    {len(df[df['Target'] == 0]):,}")

# Kiểm tra phân bố label gốc
print("\n[*] Phân bố label trong malicious:")
print(malicious_df['label'].value_counts())

# 4. LƯU TẬP MALICIOUS
output_path = '../data/malicious_ctu13.parquet'

malicious_df.to_parquet(
    output_path,
    index=False
)

print(f"\n[+] Đã lưu malicious dataset vào: {output_path}")


[+] Số mẫu malicious: 262,573
[+] Số mẫu normal:    202,549

[*] Phân bố label trong malicious:
label
flow=From-Botnet-V42-UDP-DNS                                       25361
flow=From-Botnet-V50-1-UDP-DNS                                     14404
flow=From-Botnet-V44-TCP-Attempt                                   12377
flow=From-Botnet-V50-7-UDP-DNS                                     11887
flow=From-Botnet-V50-3-UDP-DNS                                     11180
                                                                   ...  
flow=From-Botnet-V50-7-TCP-Established-HTTP-Ad-63                      1
flow=From-Botnet-V50-7-TCP-HTTP-Google-Net-Established-2               1
flow=From-Botnet-V50-7-TCP-HTTP-Not-Encrypted-Down-2                   1
flow=From-Botnet-V50-7-TCP-Established-HTTP-Ad-60                      1
flow=From-Botnet-V50-2-TCP-Established-HTTP-To-Microsoft-Live-2        1
Name: count, Length: 1263, dtype: int64

[+] Đã lưu malicious dataset vào: ../data/malicious_ct